# NCP -- Neural Circuit Policies

Lechner, Hasani, Amini, Henzinger, Rus, Grosu, *Neural circuit policies enabling auditable autonomy*, Nature MI 2020.

LTC neurons wired sparsely into `sensory -> inter -> command (recurrent) -> motor` layers instead of fully-connected. See `model.py` (`NCPWiring` + `NCPCell`) and `../../papers/README.md`.

This notebook trains an `NCPModel` on UCI Room Occupancy Detection and visualizes the sparse wiring mask.

In [ ]:
import sys
sys.path.insert(0, '../..')
sys.path.insert(0, '.')

import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from liquid_playground.data import load_room_occupancy
from liquid_playground.device import resolve_device
from liquid_playground.utils.seed import set_seed
from model import NCPModel

set_seed(0)
device = resolve_device('auto')  # or 'cpu' / 'cuda' / 'mps'
print('device:', device)

In [ ]:
train_x, train_y, test_x, test_y = load_room_occupancy()
train_x, test_x = train_x.to(device), test_x.to(device)
train_y, test_y = train_y.to(device), test_y.to(device)
print(train_x.shape)

In [ ]:
model = NCPModel(input_size=train_x.shape[-1], output_size=1, seed=0).to(device)

plt.figure(figsize=(5, 5))
plt.imshow(model.wiring.recurrent_mask.numpy(), cmap='Greys', aspect='auto')
plt.title('NCP sparse recurrent wiring mask (inter | command | motor)')
plt.xlabel('source neuron'); plt.ylabel('target neuron')
plt.show()

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.BCEWithLogitsLoss()

history = {'train_loss': [], 'test_acc': []}
epochs = 40
for epoch in range(epochs):
    model.train()
    opt.zero_grad()
    logits = model(train_x).squeeze(-1)
    loss = loss_fn(logits, train_y)
    loss.backward()
    opt.step()

    model.eval()
    with torch.no_grad():
        test_acc = ((model(test_x).squeeze(-1) > 0).float() == test_y).float().mean().item()
    history['train_loss'].append(loss.item())
    history['test_acc'].append(test_acc)

print(f"final test accuracy: {history['test_acc'][-1]:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history['train_loss']); axes[0].set_title('train loss'); axes[0].set_xlabel('epoch')
axes[1].plot(history['test_acc']); axes[1].set_title('test accuracy'); axes[1].set_xlabel('epoch')
fig.tight_layout()
plt.show()